# Cazadores de Mitos: ¿Existe realmente la Velocidad Constante? 📊

En los libros de texto de Física, el Movimiento Rectilíneo Uniforme (MRU) se describe como un movimiento perfecto donde la velocidad nunca cambia. Si graficamos la Posición vs. el Tiempo, obtendremos una línea recta perfecta. 

¿Pero qué pasa en el mundo físico real, lleno de roce, motores imperfectos y caídas de voltaje de la batería? Hoy convertiremos al Ladrillo NXT en una grabadora de telemetría para extraer sus datos internos y descubrir la verdad.


In [1]:
import random
from IPython.display import display, HTML

# UID para evitar colisiones en Jupyter Book
uid = str(random.randint(10000, 99999))

simulacion_html = """
<div style="border: 1px solid #e0e0e0; padding: 20px; border-radius: 8px; background-color: #f8f9fa; font-family: -apple-system, sans-serif;">
    <h3 style="margin-top:0; color: #2c3e50;">Simulador: Datalogging y Graficación MRU</h3>
    <p style="font-size: 0.95em; color: #555; margin-bottom: 15px;">
        Observa cómo los datos mecánicos del tacómetro se convierten en un gráfico matemático. La línea azul discontinua representa la teoría perfecta (MRU). La línea roja muestra la telemetría real del robot luchando contra la inercia y el roce.
    </p>
    
    <div style="display: flex; gap: 20px; margin-bottom: 20px; background: #e9ecef; padding: 15px; border-radius: 8px; flex-wrap: wrap; align-items: center; justify-content: space-between;">
        <div style="display: flex; gap: 10px; align-items: center;">
            <button id="play_UID" style="background-color: #0d6efd; color: white; border: none; padding: 10px 15px; border-radius: 5px; cursor: pointer; font-weight: bold; font-size: 14px;">▶ Iniciar Telemetria (5 Segundos)</button>
            <button id="reset_UID" style="background-color: #6c757d; color: white; border: none; padding: 10px 15px; border-radius: 5px; cursor: pointer; font-weight: bold; font-size: 14px;">🔄 Reiniciar Datos</button>
        </div>
        
        <div style="display: flex; flex-direction: column; align-items: center; background: #fff; padding: 10px 20px; border-radius: 5px; border: 1px solid #ccc;">
            <label style="font-weight: 600; font-size: 0.85em; color: #6f42c1; margin-bottom: 5px;">Roce e Imperfeccion del Suelo</label>
            <div style="display: flex; align-items: center; gap: 10px;">
                <input type="range" id="noise_val_UID" min="0" max="100" step="10" value="30" style="width: 150px; cursor: pointer;">
                <span id="lbl_noise_UID" style="font-weight: bold; font-family: monospace; font-size: 1.1em; width: 45px; color: #212529;">30 %</span>
            </div>
        </div>
    </div>

    <div style="display: flex; gap: 15px; justify-content: center; margin-bottom: 15px; flex-wrap: wrap;">
        <div style="background: #212529; color: #00ff00; padding: 8px 15px; border-radius: 5px; border: 1px solid #000; width: 140px; text-align: center; font-family: monospace;">
            <label style="font-size: 0.8em; margin-bottom: 2px; color: #aaa; display: block;">Tiempo (ms)</label>
            <span id="tele_time_UID" style="font-weight: bold; font-size: 1.2em;">0</span>
        </div>
        <div style="background: #212529; color: #0dcfd5; padding: 8px 15px; border-radius: 5px; border: 1px solid #000; width: 140px; text-align: center; font-family: monospace;">
            <label style="font-size: 0.8em; margin-bottom: 2px; color: #aaa; display: block;">Grados (Taco)</label>
            <span id="tele_deg_UID" style="font-weight: bold; font-size: 1.2em;">0</span>
        </div>
        <div style="background: #fff; padding: 8px 15px; border-radius: 5px; border: 1px solid #ccc; width: 180px; text-align: center;">
            <label style="font-weight: 600; font-size: 0.8em; margin-bottom: 2px; color: #dc3545; display: block;">Distancia Calculada (Excel)</label>
            <span id="tele_dist_UID" style="font-weight: bold; font-family: monospace; font-size: 1.2em; color: #212529;">0.00 cm</span>
        </div>
    </div>
    
    <div style="display: flex; justify-content: center; margin-bottom: 10px;">
        <canvas id="canvas_sim_UID" width="800" height="450" style="background: #ffffff; border: 1px solid #ccc; border-radius: 4px; box-shadow: 0 2px 5px rgba(0,0,0,0.05); max-width: 100%;"></canvas>
    </div>
</div>

<script>
    (function() {
        function initSimulation() {
            const canvas = document.getElementById('canvas_sim_UID');
            if (!canvas) {
                setTimeout(initSimulation, 50); 
                return;
            }
            
            const ctx = canvas.getContext('2d');
            const btn_play = document.getElementById('play_UID');
            const btn_reset = document.getElementById('reset_UID');
            const slider_noise = document.getElementById('noise_val_UID');
            const lbl_noise = document.getElementById('lbl_noise_UID');
            
            const tele_time = document.getElementById('tele_time_UID');
            const tele_deg = document.getElementById('tele_deg_UID');
            const tele_dist = document.getElementById('tele_dist_UID');
            
            // Constantes de Layout Canvas
            const W = canvas.width;
            const H = canvas.height;
            const Y_ROAD = 120;
            const GRAPH_X = 60;
            const GRAPH_Y = 170;
            const GRAPH_W = 700;
            const GRAPH_H = 240;
            
            // Parámetros Físicos y del Robot
            const WHEEL_DIAMETER_CM = 5.6; // Rueda estándar NXT
            const WHEEL_CIRCUMFERENCE = Math.PI * WHEEL_DIAMETER_CM; // ~17.59 cm
            const TARGET_SPEED_CM_S = 20.0; // Velocidad de crucero teórica (50% motor)
            const MAX_TIME_S = 5.0; // Experimento de 5 segundos
            
            // Estado de Simulación
            let isRunning = false;
            let lastTime = 0;
            let simTime = 0; // Tiempo del experimento en segundos
            
            let realDistCm = 0;
            let theoreticalDistCm = 0;
            let currentVelocity = 0;
            
            // Almacenamiento de datos (El archivo CSV en memoria)
            let dataLog = []; 
            let nextLogTime = 0; // Para guardar datos cada 50ms igual que NXC

            function drawRobot(x_cm) {
                // Escalar centímetros a píxeles para el dibujo superior
                // 100 cm = ancho de pantalla util (700px) -> 1cm = 7px
                let px = 50 + (x_cm * 7); 
                let y = Y_ROAD;
                
                // Ruedas
                ctx.fillStyle = "#333";
                ctx.fillRect(px + 5, y - 15, 20, 25);
                ctx.fillRect(px + 45, y - 15, 20, 25);
                
                // Chasis
                ctx.fillStyle = "#f8f9fa";
                ctx.strokeStyle = "#adb5bd";
                ctx.lineWidth = 2;
                ctx.beginPath();
                ctx.roundRect(px, y - 50, 70, 45, 5);
                ctx.fill(); ctx.stroke();
                
                // Pantalla con simbolo de grabación
                ctx.fillStyle = "#c2d1b2";
                ctx.fillRect(px + 15, y - 45, 25, 18);
                if (isRunning) {
                    ctx.fillStyle = "#dc3545"; // Punto rojo de Rec
                    ctx.beginPath(); ctx.arc(px + 27, y - 36, 4, 0, Math.PI*2); ctx.fill();
                }
            }

            function drawGraphBackground() {
                // Ejes
                ctx.strokeStyle = "#333";
                ctx.lineWidth = 2;
                ctx.beginPath();
                ctx.moveTo(GRAPH_X, GRAPH_Y);
                ctx.lineTo(GRAPH_X, GRAPH_Y + GRAPH_H); // Eje Y (Distancia)
                ctx.lineTo(GRAPH_X + GRAPH_W, GRAPH_Y + GRAPH_H); // Eje X (Tiempo)
                ctx.stroke();
                
                // Etiquetas
                ctx.fillStyle = "#555";
                ctx.font = "14px sans-serif";
                ctx.fillText("Tiempo (Segundos)", GRAPH_X + GRAPH_W/2 - 50, GRAPH_Y + GRAPH_H + 35);
                
                ctx.save();
                ctx.translate(GRAPH_X - 40, GRAPH_Y + GRAPH_H/2 + 40);
                ctx.rotate(-Math.PI/2);
                ctx.fillText("Distancia (cm)", 0, 0);
                ctx.restore();
                
                // Grilla X (0 a 5 segundos)
                ctx.font = "12px sans-serif";
                ctx.fillStyle = "#888";
                for (let s = 0; s <= 5; s++) {
                    let px = GRAPH_X + (s / MAX_TIME_S) * GRAPH_W;
                    ctx.strokeStyle = "#eee";
                    ctx.beginPath(); ctx.moveTo(px, GRAPH_Y); ctx.lineTo(px, GRAPH_Y + GRAPH_H); ctx.stroke();
                    ctx.fillText(s + "s", px - 5, GRAPH_Y + GRAPH_H + 15);
                }
                
                // Grilla Y (0 a 100 cm)
                for (let d = 0; d <= 100; d += 20) {
                    let py = GRAPH_Y + GRAPH_H - (d / 100) * GRAPH_H;
                    ctx.strokeStyle = "#eee";
                    ctx.beginPath(); ctx.moveTo(GRAPH_X, py); ctx.lineTo(GRAPH_X + GRAPH_W, py); ctx.stroke();
                    ctx.fillText(d, GRAPH_X - 25, py + 4);
                }
                
                // Leyenda
                ctx.font = "bold 12px sans-serif";
                ctx.fillStyle = "#0d6efd"; ctx.fillText("--- MRU Teorico", GRAPH_X + 20, GRAPH_Y + 20);
                ctx.fillStyle = "#dc3545"; ctx.fillText("--- Datos Reales (Taco)", GRAPH_X + 20, GRAPH_Y + 40);
            }

            function drawSim() {
                ctx.clearRect(0, 0, W, H);
                
                // 1. Dibujar Mundo Físico (Top)
                ctx.fillStyle = "#e9ecef";
                ctx.fillRect(0, Y_ROAD, W, H - Y_ROAD); // Fondo gris claro abajo de la calle para rellenar
                ctx.fillStyle = "#fff";
                ctx.fillRect(0, 0, W, Y_ROAD); // Fondo blanco arriba
                
                ctx.strokeStyle = "#adb5bd";
                ctx.lineWidth = 2;
                ctx.beginPath(); ctx.moveTo(0, Y_ROAD); ctx.lineTo(W, Y_ROAD); ctx.stroke();
                
                drawRobot(realDistCm);
                
                // 2. Dibujar Gráfico (Bottom)
                ctx.fillStyle = "#fff";
                ctx.fillRect(0, 140, W, H - 140); // Limpiar zona de grafico
                drawGraphBackground();
                
                // Dibujar línea Teórica (Azul discontinua)
                ctx.strokeStyle = "rgba(13, 110, 253, 0.6)";
                ctx.lineWidth = 3;
                ctx.setLineDash([10, 10]);
                ctx.beginPath();
                ctx.moveTo(GRAPH_X, GRAPH_Y + GRAPH_H); // Origen
                // Punto actual teórico
                let thX = GRAPH_X + (simTime / MAX_TIME_S) * GRAPH_W;
                let thY = GRAPH_Y + GRAPH_H - (theoreticalDistCm / 100) * GRAPH_H;
                ctx.lineTo(thX, thY);
                ctx.stroke();
                ctx.setLineDash([]);
                
                // Dibujar línea Real (Roja basada en DataLog)
                if (dataLog.length > 0) {
                    ctx.strokeStyle = "#dc3545";
                    ctx.lineWidth = 3;
                    ctx.beginPath();
                    
                    for (let i = 0; i < dataLog.length; i++) {
                        let pt = dataLog[i];
                        let px = GRAPH_X + (pt.timeS / MAX_TIME_S) * GRAPH_W;
                        let py = GRAPH_Y + GRAPH_H - (pt.distCm / 100) * GRAPH_H;
                        
                        if (i === 0) ctx.moveTo(px, py);
                        else ctx.lineTo(px, py);
                    }
                    ctx.stroke();
                }

                // UI y Telemetría
                lbl_noise.innerText = slider_noise.value + " %";
                
                if (dataLog.length > 0) {
                    let lastData = dataLog[dataLog.length - 1];
                    let timeMs = Math.round(lastData.timeS * 1000);
                    let grados = Math.round(lastData.grados);
                    let distVisual = lastData.distCm.toFixed(2);
                    
                    tele_time.innerText = timeMs;
                    tele_deg.innerText = grados;
                    tele_dist.innerText = distVisual + " cm";
                }
                
                if (!isRunning && simTime >= MAX_TIME_S) {
                    btn_play.innerText = "Experimento Terminado (100% CSV)";
                    btn_play.style.backgroundColor = "#198754";
                }
            }

            function updatePhysics(dt) {
                // Distancia teórica (Fórmula de libro: d = v * t)
                theoreticalDistCm = TARGET_SPEED_CM_S * simTime;
                
                // Física real: Aceleración progresiva + Ruido
                let noiseLevel = parseInt(slider_noise.value) / 100;
                
                // El robot intenta alcanzar TARGET_SPEED pero tarda por inercia
                let accel = (TARGET_SPEED_CM_S - currentVelocity) * 3.0; // Factor P de inercia
                currentVelocity += accel * dt;
                
                // Añadir ruido aleatorio basado en el slider de roce (solo si ya se está moviendo)
                if (currentVelocity > 1) {
                    let randomJitter = (Math.random() - 0.5) * 5.0 * noiseLevel; // Jitter de velocidad
                    currentVelocity += randomJitter;
                    
                    // Efecto de fricción constante aleatoria (micro frenadas)
                    if (Math.random() < 0.2) {
                        currentVelocity -= 2.0 * noiseLevel;
                    }
                }
                
                // Evitar velocidad negativa por ruido excesivo
                if (currentVelocity < 0) currentVelocity = 0;
                
                realDistCm += currentVelocity * dt;
                
                // Simulación de Datalogging (Guardar datos en la "memoria" cada 50ms = 0.05s)
                if (simTime >= nextLogTime) {
                    // Calculo matemático: Convertir cm reales a los grados del motor que leería el NXT
                    let revoluciones = realDistCm / WHEEL_CIRCUMFERENCE;
                    let gradosMotor = revoluciones * 360;
                    
                    dataLog.push({
                        timeS: simTime,
                        grados: gradosMotor,
                        distCm: realDistCm
                    });
                    nextLogTime += 0.05; // Programar próxima lectura en 50ms
                }
            }

            function animate(timestamp) {
                if (!lastTime) lastTime = timestamp;
                let dt = (timestamp - lastTime) / 1000;
                lastTime = timestamp;
                if (dt > 0.05) dt = 0.05; 
                
                if (isRunning) {
                    simTime += dt;
                    if (simTime > MAX_TIME_S) {
                        simTime = MAX_TIME_S;
                        isRunning = false; // Fin a los 5 segundos
                    }
                    updatePhysics(dt);
                }
                
                drawSim();
                requestAnimationFrame(animate);
            }

            btn_play.addEventListener('click', () => {
                if (simTime >= MAX_TIME_S) return; // Requiere reinicio
                
                isRunning = true;
                btn_play.innerText = "⏸ Grabando Datos CSV...";
                btn_play.style.backgroundColor = "#ffc107";
                btn_play.style.color = "#000";
            });

            btn_reset.addEventListener('click', () => {
                isRunning = false; 
                simTime = 0;
                lastTime = 0;
                realDistCm = 0;
                theoreticalDistCm = 0;
                currentVelocity = 0;
                dataLog = [];
                nextLogTime = 0;
                
                tele_time.innerText = "0";
                tele_deg.innerText = "0";
                tele_dist.innerText = "0.00 cm";
                
                btn_play.innerText = "▶ Iniciar Telemetria (5 Segundos)";
                btn_play.style.backgroundColor = "#0d6efd";
                btn_play.style.color = "#fff";
                drawSim();
            });

            slider_noise.addEventListener('input', () => {
                lbl_noise.innerText = slider_noise.value + " %";
            });
            
            // Iniciar frame initial
            requestAnimationFrame(animate);
        }
        
        initSimulation();
    })();
</script>
"""

html_final = simulacion_html.replace('UID', uid)
display(HTML(html_final))


## 1. Guía del Estudiante

### El Desafío
Deberás programar el robot para que avance en línea recta durante 5 segundos intentando mantener una velocidad constante del 50%. Sin embargo, la magia no estará en el movimiento, sino en la **recolección de datos**.
El robot deberá leer la información de sus propios tacómetros internos (que miden los grados de rotación de las ruedas) cada décima de segundo y guardar esa información en un archivo dentro de su memoria. Finalmente, descargaremos ese archivo para analizarlo en el computador.

### Fase 1: Ingeniería (Construcción)
1. Construye la **Base Motriz Estándar**.
2. Asegúrate de que los motores estén firmemente conectados a los **puertos B y C**. Para este experimento, no necesitamos ningún sensor externo, solo los motores inteligentes.
3. *Averigua y anota:* Mide con una regla el diámetro exacto de la rueda de tu robot (en centímetros). Necesitaremos este dato matemático más adelante para convertir los "grados del motor" en "distancia real".

### Fase 2: Programación (Código NXC)
Copia este código en BricxCC. Aquí usaremos un sistema de archivos interno. Cada vez que el bucle se ejecute, el robot tomará "una foto" de su tacómetro y la guardará en la memoria.

```c
task main() {
    // 1. Crear el archivo en la memoria del NXT
    byte archivo;            // Etiqueta del archivo
    int size = 4096;         // Tamaño del archivo en bytes (4KB es suficiente)
    string nombreArchivo = "datos_mru.csv"; // CSV = Comma Separated Values
    
    // Si el archivo ya existía de una prueba anterior, lo borramos para empezar limpio
    DeleteFile(nombreArchivo);
    
    // Creamos el archivo y lo abrimos para escritura
    CreateFile(nombreArchivo, size, archivo);
    
    // Escribimos el encabezado de nuestras columnas en el archivo
    WriteLnString(archivo, "Tiempo(ms),Grados_Motor_C", size);
    
    // 2. Preparar el experimento
    ResetTachoCount(OUT_C); // Reseteamos el contador de grados de la rueda derecha a cero
    int tiempo = 0;
    int grados = 0;
    string lineaTexto;
    
    // Arrancamos los motores al 50% de potencia
    OnFwd(OUT_BC, 50);
    
    // 3. Bucle de Recolección de Datos (Datalogging)
    long tiempoInicio = CurrentTick(); // Guardamos el milisegundo exacto en el que empezamos
    
    // Mientras no hayan pasado 5000 milisegundos (5 segundos)
    while (CurrentTick() - tiempoInicio < 5000) {
        
        // Calculamos cuánto tiempo ha pasado
        tiempo = CurrentTick() - tiempoInicio;
        
        // Leemos cuántos grados ha girado el motor C hasta este momento
        grados = MotorTachoCount(OUT_C);
        
        // Unimos el número del tiempo y el de los grados en una sola línea separada por coma
        // Ejemplo: "1500,720" (A los 1.5 segundos, el motor dio dos vueltas completas)
        lineaTexto = NumToStr(tiempo) + "," + NumToStr(grados);
        
        // Guardamos la línea de texto en nuestro archivo
        WriteLnString(archivo, lineaTexto, size);
        
        // Esperamos 50 milisegundos antes de tomar el siguiente dato
        Wait(50); 
    }
    
    // 4. Fin del experimento
    Off(OUT_BC);           // Apagamos los motores
    CloseFile(archivo);    // Cerramos y guardamos el archivo
    
    // Emitimos un sonido feliz para avisar que los datos están listos
    PlayTone(1000, 500); 
}
```

### Fase 3: Análisis de Laboratorio
1. Ejecuta el robot en el suelo (debe ir recto).
2. Conecta el NXT al PC por USB. Abre el programa **NXT Explorer** (viene incluido en las herramientas de BricxCC).
3. Busca el archivo `datos_mru.csv` en la memoria del ladrillo y arrástralo a tu escritorio.
4. Ábrelo con Excel o Google Sheets. ¡Verás cientos de datos reales tomados por tu robot!
5. **El Desafío Final:** Los datos están en *Grados*. Aplica la siguiente fórmula en tu planilla para transformar esos grados en *Distancia Recorrida en Centímetros*:
   $$ Distancia = \frac{Grados}{360} \times (\pi \times Diámetro\_de\_la\_Rueda) $$
6. Grafica la *Distancia* vs. el *Tiempo*. ¿Es una línea recta perfecta? ¿Qué pasa al principio del gráfico cuando el robot recién empieza a moverse?

---

## 2. Guía del Docente

> **Nota para el Profesor:** Este experimento demuestra cómo un microcontrolador percibe la física y transforma a los estudiantes en verdaderos científicos de datos. Es un pilar fundamental para el pensamiento científico y analítico.

### Vínculo Curricular (MINEDUC Chile)
* **Física (2º Medio):** OA 9 - Analizar, sobre la base de la experimentación, el movimiento rectilíneo uniforme y acelerado de un objeto respecto de un sistema de referencia espacial-temporal, considerando variables como la posición, la velocidad y la aceleración.
* **Matemáticas (1º a 3º Medio):** Uso de planillas de cálculo (Excel) y herramientas tecnológicas para procesar información, construir gráficos de funciones y analizar la tasa de variación.
* **Tecnología (1º Medio):** Elaborar hojas de cálculo para procesar datos, incorporar herramientas de gráficos y presentarlos visualmente.

### Conceptos Clave a Desarrollar
1. **Telemetría y Tacómetros:** El motor del NXT no es un simple motor eléctrico de juguete; es un *servomotor* con un sensor de cuadratura óptico (encoder) que tiene una resolución de 1 grado. El robot *sabe* exactamente en qué posición está su rueda en todo momento.
2. **Transformación de Datos Crudos:** Este es el concepto matemático más importante de la clase. El robot arroja datos crudos (Grados). El alumno debe conectar geometría (Perímetro de la circunferencia = $\pi \times D$) y proporción ($\frac{x}{360}$ vueltas) para convertir un dato mecánico en una magnitud de desplazamiento lineal (centímetros).
3. **El "Ruido" Físico:** En el gráfico final, los estudiantes descubrirán que la línea nunca es recta perfecta. 
   * Al inicio verán una curva (aceleración inicial mientras el motor vence la inercia).
   * Durante el viaje, verán pequeños "dientes" en la gráfica. Ese es el microprocesador interno (PID) intentando corregir la velocidad para mantener el motor C y el motor B sincronizados. Las Leyes de la Termodinámica y el roce impiden el movimiento perfecto.

### Resolución de Problemas (Troubleshooting)
* **Error de memoria llena en el NXT:** Si los alumnos olvidan poner la función `CloseFile(archivo)` al final del código, el archivo quedará corrupto y ocupará espacio fantasma en el bloque. Deberán ir al menú de configuración del ladrillo físico y eliminar los archivos no válidos.
* **Excel no separa las columnas:** En algunos computadores, Excel usa punto y coma (`;`) para separar valores en lugar de coma (`,`). Si el archivo se ve todo amontonado en la columna A, pídales que cambien la línea en NXC a: `lineaTexto = NumToStr(tiempo) + ";" + NumToStr(grados);`.
* **Desfase de tiempo inicial:** El comando `CurrentTick()` cuenta los milisegundos desde que el ladrillo NXT se encendió, no desde que inició el programa. Es vital restar el `tiempoInicio` para que la gráfica comience en el segundo cero (`tiempo = CurrentTick() - tiempoInicio`).